# Fine Tunning do BART

## 1. Instalar dependências (Colab / VSCode)

In [1]:
pip install transformers datasets accelerate

In [3]:
!pip install "transformers[torch]"
!pip install accelerate

In [4]:
import transformers
print(transformers.__version__)

5.7.0


## 2. Carregar dataset

In [5]:
import json
from datasets import Dataset

with open("dataset_final.json", "r", encoding="utf-8") as f:
    data = json.load(f)

dataset = Dataset.from_list(data)

print(dataset[0])

{'input': 'what genres are the films written by Frederick Hazlitt Brennan in', 'output': 'Frederick Hazlitt Brennan -> written_by_inv -> A Guy Named Joe -> has_genre -> War'}


## 3. Split treino / validação

In [6]:
dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
val_dataset = dataset["test"]

## 4. Tokenização

In [7]:
from transformers import BartTokenizer

tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")

MAX_INPUT = 64
MAX_OUTPUT = 128

def preprocess(example):
    model_input = tokenizer(
        example["input"],
        max_length=MAX_INPUT,
        truncation=True,
        padding="max_length"
    )
    
    labels = tokenizer(
        example["output"],
        max_length=MAX_OUTPUT,
        truncation=True,
        padding="max_length"
    )

    model_input["labels"] = labels["input_ids"]
    return model_input

train_dataset = train_dataset.map(preprocess, batched=False)
val_dataset = val_dataset.map(preprocess, batched=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Map:   0%|          | 0/52608 [00:00<?, ? examples/s]

Map:   0%|          | 0/5846 [00:00<?, ? examples/s]

## 5. Carregar modelo

In [8]:
from transformers import BartForConditionalGeneration

model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

## 6. Configuração de treino

In [11]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    fp16=True,  # usar GPU
    logging_steps=100
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


## 7. Trainer

In [12]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

## 8. Treinar

In [13]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.152124,0.139099
2,0.129712,0.122697
3,0.114391,0.116152


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=19728, training_loss=0.16062398642809142, metrics={'train_runtime': 1231.6256, 'train_samples_per_second': 128.143, 'train_steps_per_second': 16.018, 'total_flos': 6014440424079360.0, 'train_loss': 0.16062398642809142, 'epoch': 3.0})

## 9. Salvar modelo

In [14]:
trainer.save_model("bart_paths_model")
tokenizer.save_pretrained("bart_paths_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bart_paths_model/tokenizer_config.json', 'bart_paths_model/tokenizer.json')

## 10. Teste rápido

In [15]:
def predict(question):
    inputs = tokenizer(question, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_length=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(predict("who directed movies written by Quentin Tarantino"))

Quentin Tarantino -> written_by_inv -> Tarantula -> directed_by -> Quentin Tarantino


# Download do Modelo

In [19]:
!zip -r bart_paths_model.zip bart_paths_model

updating: bart_paths_model/ (stored 0%)
updating: bart_paths_model/training_args.bin (deflated 53%)
updating: bart_paths_model/model.safetensors (deflated 8%)
updating: bart_paths_model/config.json (deflated 65%)
updating: bart_paths_model/tokenizer_config.json (deflated 50%)
updating: bart_paths_model/tokenizer.json (deflated 82%)
updating: bart_paths_model/generation_config.json (deflated 47%)


In [20]:
from google.colab import files
files.download("bart_paths_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
!zip -r bart_paths_model.zip bart_paths_model
!split -b 100M bart_paths_model.zip part_

  adding: bart_paths_model/ (stored 0%)
  adding: bart_paths_model/training_args.bin (deflated 53%)
  adding: bart_paths_model/model.safetensors (deflated 8%)
  adding: bart_paths_model/config.json (deflated 65%)
  adding: bart_paths_model/tokenizer_config.json (deflated 50%)
  adding: bart_paths_model/tokenizer.json (deflated 82%)
  adding: bart_paths_model/generation_config.json (deflated 47%)
